# O-RAN Version 4 — Dual-Uncertainty QoS Safety Shield

V4 investigates whether uncertainty-calibrated traffic targets and receiver-specific rate lower bounds can filter PPO power actions under imperfect CSI. It is a research hypothesis, not a verified novelty or safety guarantee.

## Mathematical idea

Traffic: d_upper,k = clip(d_hat,k + q_d,k, 0, 1), where q_d,k is a one-sided split-conformal residual quantile from a disjoint calibration set.

QoS target: R_min,k = R_base + R_span d_upper,k.

For receiver r, draw posterior channel scenarios around H_hat and compute a lower-tail rate L_k,r(p). Calibrate a receiver/user margin q_c,k,r on independent calibration actions, then define LCB_k,r(p) = L_k,r(p) - q_c,k,r.

The runtime shield attempts to solve min ||p-p_PPO||^2 subject to sum(p) <= P_max, p >= 0 and LCB_k,r(p) >= R_min,k. These constraints are coupled and non-convex; SLSQP may fail.

## Run the full implementation

Download and extract `O-RAN-Version4-Dual-Uncertainty-QoS-Safe-DRL.zip` from the Version 4 project bundle into a working folder. Open a terminal in the extracted `O-RAN-Version4` directory.

In [ ]:
from pathlib import Path
import subprocess, sys
V4 = Path.cwd()
if not (V4/'src'/'safety.py').exists():
    raise FileNotFoundError('Open this notebook with the extracted O-RAN-Version4 folder as the working directory.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


## Train demand model and conformal upper bounds

In [ ]:
subprocess.run([sys.executable, '-m', 'src.train_demand', '--epochs', '20'], check=True)


## Calibrate receiver-specific channel-rate margins

In [ ]:
subprocess.run([sys.executable, '-m', 'src.calibrate', '--samples', '240'], check=True)


## Train receiver-specific PPO controllers

In [ ]:
for rx in ('MRC', 'ZF', 'MMSE'):
    subprocess.run([sys.executable, '-m', 'src.train_ppo', '--receiver', rx], check=True)


## Evaluate raw PPO and uncertainty shields

In [ ]:
subprocess.run([sys.executable, '-m', 'src.evaluate', '--seeds', '5', '--blocks', '50'], check=True)
import pandas as pd
display(pd.read_csv('results/summary_ci95.csv'))


## Research limitations

The smoke run used only 2 PPO episodes × 5 steps, 12 channel-calibration samples and 2 test seeds × 2 blocks. It passed 4 unit tests but is not research-grade validation. The demand MLP did not beat persistence in the reported 20-epoch synthetic split. Conformal coverage is marginal and assumes exchangeability; it is not a deterministic or simultaneous all-user guarantee. SLSQP is local and may fail. No live O-RAN, A1/E2 or measured hardware power implementation is included.